# Preprocessing

In [21]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from src import config

print("✅ Setup complete.")

✅ Setup complete.


In [22]:
# Generic loading mechanism
INTERIM_DATA_PATH = '../data/interim/water_quality_mvp_baseline.parquet' 
df = pd.read_parquet(INTERIM_DATA_PATH)

print(f"Loaded shape: {df.shape}")
display(df.head(3))

Loaded shape: (9319, 10)


,Latitude,Longitude,Sample Date,swir22,NDMI,MNDWI,pet,Total Alkalinity,Electrical Conductance,Dissolved Reactive Phosphorus
0,-28.760833,17.730278,2011-01-02,7645.0,0.185538,0.195595,174.2,128.912,555.0,10.0
1,-26.861111,28.884722,2011-01-03,10574.0,0.124566,-0.180134,124.1,74.720,162.9,163.0
2,-26.450000,28.085833,2011-01-03,14201.0,-0.083293,-0.252805,127.5,89.254,573.0,80.0


### Assign Region

In [23]:
def assign_region(row):
    # Using the bounding boxes discovered during EDA
    if row['Latitude'] >= -30.0:
        return 'Northern_Bulk'
    elif row['Longitude'] < 22.5:
        return 'Western_Cape'
    else:
        return 'Eastern_Cape'

df['Region'] = df.apply(assign_region, axis=1)

print("\nData distribution across regions:")
print(df['Region'].value_counts())


Data distribution across regions:
Region
Northern_Bulk    6950
Western_Cape     1289
Eastern_Cape     1080
Name: count, dtype: int64


### Drop Spatial Anchor

In [24]:
# --- Drop Spatial Anchors ---
# We drop the raw GPS coordinates so the model CANNOT use them, 
# but we keep our new 'Region' column to split the folds later.
META_COLS = ['Latitude', 'Longitude', 'Sample Date']
df = df.drop(columns=META_COLS)

print(f"\nShape after dropping spatial anchors: {df.shape}")
print(f"Columns remaining: {df.columns.tolist()}")


Shape after dropping spatial anchors: (9319, 8)
Columns remaining: ['swir22', 'NDMI', 'MNDWI', 'pet', 'Total Alkalinity', 'Electrical Conductance', 'Dissolved Reactive Phosphorus', 'Region']


### Save Dataset for LORO CV

In [25]:
os.makedirs('../data/processed', exist_ok=True)

# Save the un-scaled, un-imputed dataset with the Region tags
OUTPUT_PATH = '../data/processed/data_with_regions.parquet'
df.to_parquet(OUTPUT_PATH, index=False)

print(f"✅ Data ready for LORO Cross-Validation saved to: {OUTPUT_PATH}")
print("Note: Preprocessing (Scaling/Imputation) moved to 03_model_training to prevent fold leakage.")

✅ Data ready for LORO Cross-Validation saved to: ../data/processed/data_with_regions.parquet
Note: Preprocessing (Scaling/Imputation) moved to 03_model_training to prevent fold leakage.
